# SL Neural Network Calibration
Trains MNIST / CIFAR-10 models **incrementally** (one continuous run), saves
predictions at milestone epochs before and after temperature scaling, then
analyses calibration-based trust using Subjective Logic.

> **Runtime**: go to *Runtime → Change runtime type → T4 GPU* before running.

In [ ]:
# ── GPU check ────────────────────────────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'GPU detected: {gpus[0].name}')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')
print(f'TensorFlow {tf.__version__}')

# ── Imports ───────────────────────────────────────────────────────────────────
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.ticker as ticker
from fractions import Fraction
from scipy.optimize import minimize
from tensorflow.keras import layers, models

In [ ]:
# ── Optional: mount Google Drive to persist results across sessions ───────────
# Skip this cell if temporary /content/ storage is enough.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  ← edit here before running
# ══════════════════════════════════════════════════════════════════════════════

DATASET      = 'cifar10'   # 'mnist'  or  'cifar10'
TOTAL_EPOCHS = 100

# Prediction output directory.
# To persist across sessions, point to Drive, e.g.:
#   PRED_DIR = '/content/drive/MyDrive/SL_Calibration/CIFAR10_PRED'
PRED_DIR = f'/content/{DATASET.upper()}_PRED'
FIG_DIR  = '/content/figures'

N_CLUSTERS = 10
BATCH_SIZE = 64    # 64-128 recommended on T4

print(f'Dataset      : {DATASET}')
print(f'Total epochs : {TOTAL_EPOCHS}')
print(f'Pred dir     : {PRED_DIR}')
print(f'Fig dir      : {FIG_DIR}')

## TrustOpinion

In [ ]:
class TrustOpinion:
    def __init__(self, trust_mass, distrust_mass, untrust_mass, base_rate=0.5):
        assert trust_mass >= 0 and distrust_mass >= 0 and untrust_mass >= 0
        assert round(trust_mass + distrust_mass + untrust_mass, 10) == 1
        assert 0 <= base_rate <= 1
        self.t = trust_mass
        self.d = distrust_mass
        self.u = untrust_mass
        self.a = base_rate

    def ev2tdu(pos_ev, neg_ev, W=2):
        b = pos_ev / (pos_ev + neg_ev + W)
        d = neg_ev / (pos_ev + neg_ev + W)
        u = W       / (pos_ev + neg_ev + W)
        return TrustOpinion(b, d, u)

    def vacuous():
        return TrustOpinion(0, 0, 1)

    def ftrust():
        return TrustOpinion(1, 0, 0)

    def dtrust():
        return TrustOpinion(0, 1, 0)

    def projected_prob(self, frac=False):
        if frac:
            return Fraction(self.t + self.a * self.u).limit_denominator()
        return round(self.t + self.a * self.u, 3)

    def __repr__(self):
        return f'({round(self.t,3)}, {round(self.d,3)}, {round(self.u,3)}, {round(self.a,3)})'

    def cumFuse(op1, op2):
        b1, d1, u1, a1 = op1.t, op1.d, op1.u, op1.a
        b2, d2, u2, a2 = op2.t, op2.d, op2.u, op2.a
        if b1 != 0 or b2 != 0:
            b = (b1 * u2 + b2 * u1) / (u1 + u2 - u1 * u2)
            u = u1 * u2 / (u1 + u2 - u1 * u2)
            if u1 != 1 or u2 != 1:
                a = (a1 * u2 + a2 * u1 - (a1 + a2) * u1 * u2) / (u1 + u2 - 2 * u1 * u2)
            else:
                a = (a1 + a2) / 2
        else:
            b = 0.5 * (b1 + b2)
            u = 0
            a = 0.5 * (a1 + a2)
        d = 1 - u - b
        b = round(b, 2); d = round(d, 2); u = round(u, 2); a = round(a, 2)
        if b + d + u != 1:
            u = 1 - (b + d)
        return TrustOpinion(b, d, u, a)

    def avFuse(op1, op2):
        b_A, u_A, a_A = op1.t, op1.u, op1.a
        b_B, u_B, a_B = op2.t, op2.u, op2.a
        if u_A != 0 or u_B != 0:
            b = (b_A * u_B + b_B * u_A) / (u_A + u_B)
            u = (2 * u_A * u_B) / (u_A + u_B)
            a = (a_A + a_B) / 2
        else:
            b = 0.5 * (b_A + b_B)
            u = 0
            a = (a_A + a_B) / 2
        return TrustOpinion(b, 1 - (b + u), u, a)

print('TrustOpinion ready.')

## Analysis utilities

In [ ]:
COLORS_10      = cm.viridis(np.linspace(0, 1, 10))
LABEL_FONTSIZE = 12
TITLE_FONTSIZE = 13
plt.rcParams.update({'font.size': LABEL_FONTSIZE})


def make_representatives(n_clusters):
    step = 1.0 / n_clusters
    half = step / 2.0
    reps = [round(half + i * step, 6) for i in range(n_clusters)]
    return reps, half


def compute_opinion_for_class(class_value, reps, half_step, data):
    arr_l = data['True Label'].values
    arr_p = data[f'Class_{class_value}_Probability'].values
    total_pos, total_neg = 0, 0
    for rp in reps:
        mask = (arr_p >= rp - half_step) & (arr_p < rp + half_step)
        n_i  = mask.sum()
        t_i  = int((arr_l[mask] == class_value).sum())
        total_pos += t_i
        total_neg += int(round(abs(t_i - n_i * rp)))
    return TrustOpinion.ev2tdu(total_pos, total_neg)


def compute_global_opinion(data, n_clusters=10):
    reps, half = make_representatives(n_clusters)
    n_classes  = sum(1 for c in data.columns
                     if c.startswith('Class_') and c.endswith('_Probability'))
    fused = None
    for c in range(n_classes):
        op = compute_opinion_for_class(c, reps, half, data)
        fused = op if fused is None else TrustOpinion.cumFuse(fused, op)
    return fused


def build_cluster_lookup(data, n_clusters=10):
    reps, half = make_representatives(n_clusters)
    n_classes  = sum(1 for c in data.columns
                     if c.startswith('Class_') and c.endswith('_Probability'))
    lookup = {}
    for c in range(n_classes):
        arr_l = data['True Label'].values
        arr_p = data[f'Class_{c}_Probability'].values
        for rp in reps:
            mask = (arr_p >= rp - half) & (arr_p < rp + half)
            n_i  = mask.sum()
            t_i  = int((arr_l[mask] == c).sum())
            lookup[(c, rp)] = TrustOpinion.ev2tdu(t_i, int(round(abs(t_i - n_i * rp))))
    return lookup


def get_per_prediction_trust(data, lookup, n_clusters=10):
    reps, half   = make_representatives(n_clusters)
    reps_arr     = np.array(reps)
    n_classes    = sum(1 for c in data.columns
                       if c.startswith('Class_') and c.endswith('_Probability'))
    prob_cols    = [f'Class_{i}_Probability' for i in range(n_classes)]
    probs        = data[prob_cols].values
    true_labels  = data['True Label'].values
    pred_classes = np.argmax(probs, axis=1)
    confidences  = probs[np.arange(len(probs)), pred_classes]
    beliefs, disbeliefs, uncertainties = [], [], []
    for pred_c, conf in zip(pred_classes, confidences):
        nearest_rp = reps_arr[np.argmin(np.abs(reps_arr - conf))]
        op = lookup.get((pred_c, nearest_rp), TrustOpinion.vacuous())
        beliefs.append(op.t); disbeliefs.append(op.d); uncertainties.append(op.u)
    return {
        'belief':      np.array(beliefs),
        'disbelief':   np.array(disbeliefs),
        'uncertainty': np.array(uncertainties),
        'confidence':  confidences,
        'correct':     (pred_classes == true_labels),
    }


def compute_ece(data, n_bins=10):
    n_classes   = sum(1 for c in data.columns
                      if c.startswith('Class_') and c.endswith('_Probability'))
    prob_cols   = [f'Class_{i}_Probability' for i in range(n_classes)]
    probs       = data[prob_cols].values
    true_labels = data['True Label'].values
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies  = (predictions == true_labels).astype(float)
    bin_edges   = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n   = len(true_labels)
    for i in range(n_bins):
        mask = (confidences > bin_edges[i]) & (confidences <= bin_edges[i + 1])
        if mask.sum() > 0:
            ece += (mask.sum() / n) * abs(accuracies[mask].mean() - confidences[mask].mean())
    return float(ece)


def list_epoch_files(directory):
    pattern = re.compile(r'^bef_(\d+)\.csv$')
    epochs  = []
    for fname in os.listdir(directory):
        m = pattern.match(fname)
        if m:
            e   = int(m.group(1))
            bef = os.path.join(directory, f'bef_{e}.csv')
            aft = os.path.join(directory, f'aft_{e}.csv')
            if os.path.isfile(aft):
                epochs.append((e, bef, aft))
    return sorted(epochs, key=lambda x: x[0])


def load_loss_history(directory):
    path = os.path.join(directory, 'loss_history.npy')
    if os.path.isfile(path):
        return np.load(path, allow_pickle=True).item()
    return None


print('Analysis utilities ready.')

## Model builders

In [ ]:
def build_mnist_model():
    model = models.Sequential([
        layers.Flatten(input_shape=(28, 28)),
        layers.Dense(128, activation='relu'),
        layers.Dense(10),
    ])
    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'],
    )
    return model


def build_cifar10_model(total_epochs=100):
    """WideResNet-28-4 — targets ~94-95 % on CIFAR-10."""
    depth, width = 28, 4
    n   = (depth - 4) // 6
    wd  = 5e-4
    reg = tf.keras.regularizers.l2(wd)

    def res_block(x, filters, stride=1):
        shortcut = x
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        if stride != 1 or int(shortcut.shape[-1]) != filters:
            shortcut = layers.Conv2D(
                filters, 1, strides=stride, padding='same',
                use_bias=False, kernel_regularizer=reg)(x)
        x = layers.Conv2D(
            filters, 3, strides=stride, padding='same',
            use_bias=False, kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Conv2D(
            filters, 3, padding='same',
            use_bias=False, kernel_regularizer=reg)(x)
        return layers.Add()([x, shortcut])

    inputs = tf.keras.Input(shape=(32, 32, 3))
    x = layers.RandomFlip('horizontal')(inputs)
    x = layers.RandomTranslation(0.125, 0.125, fill_mode='reflect')(x)
    x = layers.Conv2D(16, 3, padding='same', use_bias=False, kernel_regularizer=reg)(x)

    for _ in range(n):
        x = res_block(x, 16 * width)
    x = res_block(x, 32 * width, stride=2)
    for _ in range(n - 1):
        x = res_block(x, 32 * width)
    x = res_block(x, 64 * width, stride=2)
    for _ in range(n - 1):
        x = res_block(x, 64 * width)

    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, kernel_regularizer=reg)(x)

    model = tf.keras.Model(inputs, outputs)

    steps_per_epoch = 45000 // BATCH_SIZE
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.1,
        decay_steps=total_epochs * steps_per_epoch,
        alpha=1e-6,
    )
    model.compile(
        optimizer=tf.keras.optimizers.SGD(
            learning_rate=lr_schedule, momentum=0.9, nesterov=True),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'],
    )
    return model


print('Model builders ready.')

## Training utilities

In [ ]:
def softmax_with_temperature(logits, T):
    exp_logits = np.exp(logits / T)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def nll_with_temperature(T, logits, labels):
    probs = softmax_with_temperature(logits, T)
    return -np.mean(np.log(probs[np.arange(len(labels)), labels] + 1e-12))


def find_optimal_temperature(logits, labels):
    result = minimize(nll_with_temperature, x0=1.0,
                      args=(logits, labels), bounds=[(0.1, 10)])
    return float(result.x[0])


def save_predictions(probs, labels, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    n_classes = probs.shape[1]
    df = pd.DataFrame(probs, columns=[f'Class_{i}_Probability' for i in range(n_classes)])
    df['True Label'] = labels
    df.to_csv(path, index=False)


def train_and_save(dataset, outdir, total_epochs=100, val_split=0.1, batch_size=32):
    """
    Train one model incrementally for total_epochs, saving predictions at each
    milestone (epochs 1-9 every epoch, then 10, 20, ..., total_epochs).
    """
    # ── Load data ─────────────────────────────────────────────────────────
    if dataset == 'mnist':
        (x_full, y_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
        x_full = x_full / 255.0;  x_test = x_test / 255.0
        model  = build_mnist_model()
    elif dataset == 'cifar10':
        (x_full, y_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
        x_full = x_full / 255.0;  x_test = x_test / 255.0
        y_full = y_full.flatten();  y_test = y_test.flatten()
        model  = build_cifar10_model(total_epochs=total_epochs)
    else:
        raise ValueError(f'Unknown dataset: {dataset}')

    n_val    = int(len(x_full) * val_split)
    x_val,   y_val   = x_full[:n_val],  y_full[:n_val]
    x_train, y_train = x_full[n_val:],  y_full[n_val:]
    os.makedirs(outdir, exist_ok=True)

    milestones       = list(range(1, 10)) + list(range(10, total_epochs + 1, 10))
    loss_history     = {'train': [], 'val': []}
    cumulative_epoch = 0

    for milestone in milestones:
        epochs_to_run = milestone - cumulative_epoch
        if epochs_to_run <= 0:
            continue

        history = model.fit(
            x_train, y_train,
            epochs=epochs_to_run,
            batch_size=batch_size,
            validation_data=(x_val, y_val),
            verbose=1,
        )
        loss_history['train'].extend(history.history['loss'])
        loss_history['val'].extend(history.history['val_loss'])
        cumulative_epoch = milestone

        logits_test = model.predict(x_test, verbose=0)
        T_opt       = find_optimal_temperature(logits_test, y_test)
        print(f'[epoch {milestone:3d}] optimal T = {T_opt:.4f}')

        probs_bef = tf.nn.softmax(logits_test, axis=1).numpy()
        probs_aft = softmax_with_temperature(logits_test, T_opt)

        save_predictions(probs_bef, y_test, os.path.join(outdir, f'bef_{milestone}.csv'))
        save_predictions(probs_aft, y_test, os.path.join(outdir, f'aft_{milestone}.csv'))
        print(f'  Saved bef_{milestone}.csv  /  aft_{milestone}.csv')

    np.save(os.path.join(outdir, 'loss_history.npy'), loss_history)
    print('\nTraining complete. Loss history saved.')


print('Training utilities ready.')

## Run training
**Estimated time** on T4 GPU:
- MNIST (100 epochs): ~10 min
- CIFAR-10 WideResNet-28-4 (100 epochs): ~1.5 h

In [ ]:
train_and_save(DATASET, PRED_DIR, total_epochs=TOTAL_EPOCHS, batch_size=BATCH_SIZE)

## Plotting functions

In [ ]:
def plot_trust_all(directory, outdir, dataset_name, n_clusters=10):
    epoch_data = list_epoch_files(directory)
    if not epoch_data:
        print(f'No files in {directory}'); return
    reps, half = make_representatives(n_clusters)
    n_classes  = 10;  epochs = []
    per_class_bef = {k: [[] for _ in range(n_classes)] for k in 'bdu'}
    per_class_aft = {k: [[] for _ in range(n_classes)] for k in 'bdu'}
    for epoch, bef_path, aft_path in epoch_data:
        d_bef = pd.read_csv(bef_path);  d_aft = pd.read_csv(aft_path)
        for c in range(n_classes):
            ob = compute_opinion_for_class(c, reps, half, d_bef)
            oa = compute_opinion_for_class(c, reps, half, d_aft)
            per_class_bef['b'][c].append(ob.t); per_class_bef['d'][c].append(ob.d); per_class_bef['u'][c].append(ob.u)
            per_class_aft['b'][c].append(oa.t); per_class_aft['d'][c].append(oa.d); per_class_aft['u'][c].append(oa.u)
        epochs.append(epoch)
    fig, axs = plt.subplots(2, 3, figsize=(13, 6))
    col_map = [('b', 'Trust'), ('d', 'DisTrust'), ('u', 'Uncertainty')]
    for row, (per_class,) in enumerate([(per_class_bef,), (per_class_aft,)]):
        for col, (mk, col_lbl) in enumerate(col_map):
            ax = axs[row][col]
            for c in range(n_classes):
                ax.plot(epochs, per_class[mk][c], color=COLORS_10[c], linewidth=1.0)
            ax.set_ylabel(col_lbl)
            if row == 1: ax.set_xlabel('Epochs')
            if col < 2:  ax.set_ylim(0, 1)
            if col == 2: ax.ticklabel_format(axis='y', style='sci', scilimits=(-2, -2))
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=1, vmax=10))
    sm.set_array([]);  fig.subplots_adjust(right=0.92)
    cbar_ax = fig.add_axes([0.94, 0.15, 0.005, 0.7])
    fig.colorbar(sm, cax=cbar_ax)
    fig.text(0.5, 0.90, 'Before Calibration', ha='center', fontsize=14)
    fig.text(0.5, 0.47, 'After Calibration',  ha='center', fontsize=14)
    os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, f'{dataset_name}_ALL.pdf'), bbox_inches='tight')
    plt.show();  plt.close(fig)
    print(f'Saved {dataset_name}_ALL.pdf')


def plot_loss_curves(directory, outdir, dataset_name):
    history = load_loss_history(directory)
    if history is None:
        print(f'loss_history.npy not found in {directory}'); return
    epochs = np.arange(1, len(history['train']) + 1)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(epochs, history['train'], label='Training',   color='steelblue', linewidth=1.8)
    ax.plot(epochs, history['val'],   label='Validation', color='firebrick', linewidth=1.8, linestyle='--')
    ax.set_xlabel('Epoch');  ax.set_ylabel('Cross-entropy loss')
    ax.set_title(f'{dataset_name} — Training and Validation Loss')
    ax.legend();  ax.grid(linestyle='--', alpha=0.4)
    os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, f'{dataset_name}_OV.pdf'), bbox_inches='tight')
    plt.show();  plt.close(fig)
    print(f'Saved {dataset_name}_OV.pdf')


def plot_trust_and_ece(directory, outdir, dataset_name, n_clusters=10):
    epoch_data = list_epoch_files(directory)
    if not epoch_data:
        print(f'No files in {directory}'); return
    reps, half = make_representatives(n_clusters);  n_classes = 10;  epochs = []
    metrics_bef   = {'b': [], 'd': [], 'u': [], 'ece': []}
    metrics_aft   = {'b': [], 'd': [], 'u': [], 'ece': []}
    per_class_bef = {k: [[] for _ in range(n_classes)] for k in 'bdu'}
    per_class_aft = {k: [[] for _ in range(n_classes)] for k in 'bdu'}
    for epoch, bef_path, aft_path in epoch_data:
        d_bef = pd.read_csv(bef_path);  d_aft = pd.read_csv(aft_path)
        ob = compute_global_opinion(d_bef, n_clusters);  oa = compute_global_opinion(d_aft, n_clusters)
        metrics_bef['b'].append(ob.t); metrics_bef['d'].append(ob.d); metrics_bef['u'].append(ob.u); metrics_bef['ece'].append(compute_ece(d_bef))
        metrics_aft['b'].append(oa.t); metrics_aft['d'].append(oa.d); metrics_aft['u'].append(oa.u); metrics_aft['ece'].append(compute_ece(d_aft))
        for c in range(n_classes):
            cb = compute_opinion_for_class(c, reps, half, d_bef);  ca = compute_opinion_for_class(c, reps, half, d_aft)
            per_class_bef['b'][c].append(cb.t); per_class_bef['d'][c].append(cb.d); per_class_bef['u'][c].append(cb.u)
            per_class_aft['b'][c].append(ca.t); per_class_aft['d'][c].append(ca.d); per_class_aft['u'][c].append(ca.u)
        epochs.append(epoch)
    fig, axs = plt.subplots(2, 4, figsize=(16, 6), gridspec_kw={'wspace': 0.35, 'hspace': 0.45})
    col_labels = ['Belief (Trust)', 'Disbelief', 'Uncertainty', 'ECE']
    for row, (metrics, per_class, row_lbl) in enumerate([
            (metrics_bef, per_class_bef, 'Before Calibration'),
            (metrics_aft, per_class_aft, 'After Calibration')]):
        for col, (mk, col_lbl) in enumerate(zip('bdu', col_labels[:3])):
            ax = axs[row][col]
            for c in range(n_classes):
                ax.plot(epochs, per_class[mk][c], color=COLORS_10[c], linewidth=1.0, alpha=0.9)
            ax.set_xlabel('Epoch', fontsize=LABEL_FONTSIZE);  ax.set_ylabel(col_lbl, fontsize=LABEL_FONTSIZE)
            if mk != 'u': ax.set_ylim(0, 1)
            if row == 0:  ax.set_title(col_lbl, fontsize=TITLE_FONTSIZE)
        ax_ece = axs[row][3]
        ax_ece.plot(epochs, metrics['ece'], color='steelblue', linewidth=2.0, marker='o', markersize=3)
        ax_ece.set_xlabel('Epoch', fontsize=LABEL_FONTSIZE);  ax_ece.set_ylabel('ECE', fontsize=LABEL_FONTSIZE)
        ax_ece.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.3f}'))
        if row == 0: ax_ece.set_title('ECE', fontsize=TITLE_FONTSIZE)
        axs[row][0].annotate(row_lbl, xy=(-0.35, 0.5), xycoords='axes fraction',
                             rotation=90, va='center', ha='center', fontsize=TITLE_FONTSIZE, fontweight='bold')
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=0, vmax=9));  sm.set_array([])
    cbar = fig.colorbar(sm, ax=axs, location='right', fraction=0.015, pad=0.01)
    cbar.set_label('Class', fontsize=LABEL_FONTSIZE);  cbar.set_ticks(np.arange(0, 10))
    os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, f'{dataset_name}_ECE.pdf'), bbox_inches='tight')
    plt.show();  plt.close(fig)
    print(f'Saved {dataset_name}_ECE.pdf')


def plot_cluster_variation(directory, outdir, dataset_name, m_values=None):
    if m_values is None:
        m_values = [2, 3, 5, 8, 10, 15, 20, 30, 50, 75, 100, 150, 200, 300, 400, 500, 1000]
    epoch_data = list_epoch_files(directory)
    if not epoch_data:
        print(f'No files in {directory}'); return
    _, bef_path, aft_path = epoch_data[-1]
    d_bef = pd.read_csv(bef_path);  d_aft = pd.read_csv(aft_path)
    def sweep(data):
        bs, ds, us = [], [], []
        for m in m_values:
            op = compute_global_opinion(data, n_clusters=m)
            bs.append(op.t); ds.append(op.d); us.append(op.u)
        return np.array(bs), np.array(ds), np.array(us)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={'wspace': 0.3})
    for ax, (bs, ds, us), title in zip(
            axs, [sweep(d_bef), sweep(d_aft)],
            ['Before Calibration', 'After Calibration']):
        ax.plot(m_values, bs, 'o-',  color='steelblue', label='Belief',     linewidth=1.8, markersize=5)
        ax.plot(m_values, ds, 's--', color='firebrick', label='Disbelief',   linewidth=1.8, markersize=5)
        ax.plot(m_values, us, '^:',  color='seagreen',  label='Uncertainty', linewidth=1.8, markersize=5)
        ax.set_xlabel('Number of clusters $M$', fontsize=LABEL_FONTSIZE)
        ax.set_ylabel('Opinion component', fontsize=LABEL_FONTSIZE)
        ax.set_title(title, fontsize=TITLE_FONTSIZE)
        ax.set_xticks(m_values);  ax.set_xticklabels([str(m) for m in m_values], rotation=45)
        ax.legend(fontsize=LABEL_FONTSIZE - 1);  ax.grid(axis='y', linestyle='--', alpha=0.4);  ax.set_ylim(0, 1)
    fig.suptitle(f'{dataset_name} — Trust opinion vs. number of clusters (last epoch)', fontsize=TITLE_FONTSIZE)
    os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, f'{dataset_name}_clusters.pdf'), bbox_inches='tight')
    plt.show();  plt.close(fig)
    print(f'Saved {dataset_name}_clusters.pdf')


def plot_dynamic_assessment(directory, outdir, dataset_name, n_clusters=10):
    epoch_data = list_epoch_files(directory)
    if not epoch_data:
        print(f'No files in {directory}'); return
    last_epoch, _, aft_path = epoch_data[-1]
    d_aft  = pd.read_csv(aft_path)
    lookup = build_cluster_lookup(d_aft, n_clusters=n_clusters)
    result = get_per_prediction_trust(d_aft, lookup, n_clusters=n_clusters)
    belief = result['belief'];  disbelief = result['disbelief']
    uncertainty = result['uncertainty'];  confidence = result['confidence'];  correct = result['correct']
    n_bins = n_clusters;  bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    mean_b, mean_d, mean_u = [], [], []
    for i in range(n_bins):
        mask = (confidence > bin_edges[i]) & (confidence <= bin_edges[i + 1])
        if mask.sum() > 0:
            mean_b.append(belief[mask].mean()); mean_d.append(disbelief[mask].mean()); mean_u.append(uncertainty[mask].mean())
        else:
            mean_b.append(np.nan); mean_d.append(np.nan); mean_u.append(np.nan)
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.5), gridspec_kw={'wspace': 0.35})
    ax = axs[0]
    colors_sc = np.where(correct, '#2196F3', '#F44336')
    ax.scatter(confidence, belief, c=colors_sc, s=8, alpha=0.4, rasterized=True)
    from matplotlib.lines import Line2D
    ax.legend(handles=[
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#2196F3', markersize=7, label='Correct'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#F44336', markersize=7, label='Incorrect'),
    ], fontsize=LABEL_FONTSIZE - 1)
    ax.set_xlabel('Predicted confidence', fontsize=LABEL_FONTSIZE);  ax.set_ylabel('Trust belief $b$', fontsize=LABEL_FONTSIZE)
    ax.set_title('Confidence vs. Trust Belief', fontsize=TITLE_FONTSIZE)
    ax.set_xlim(-0.02, 1.02);  ax.set_ylim(0, 1);  ax.grid(linestyle='--', alpha=0.3)
    ax = axs[1]
    bins = np.linspace(0, 1, 21)
    ax.hist(belief[correct],  bins=bins, density=True, alpha=0.6, color='#2196F3', label='Correct',   edgecolor='none')
    ax.hist(belief[~correct], bins=bins, density=True, alpha=0.6, color='#F44336', label='Incorrect', edgecolor='none')
    ax.set_xlabel('Trust belief $b$ per prediction', fontsize=LABEL_FONTSIZE);  ax.set_ylabel('Density', fontsize=LABEL_FONTSIZE)
    ax.set_title('Distribution of Per-Prediction Belief', fontsize=TITLE_FONTSIZE)
    ax.legend(fontsize=LABEL_FONTSIZE - 1);  ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax = axs[2]
    ax.plot(bin_centres, mean_b, 'o-',  color='steelblue', linewidth=1.8, markersize=5, label='Belief')
    ax.plot(bin_centres, mean_d, 's--', color='firebrick', linewidth=1.8, markersize=5, label='Disbelief')
    ax.plot(bin_centres, mean_u, '^:',  color='seagreen',  linewidth=1.8, markersize=5, label='Uncertainty')
    ax.set_xlabel('Predicted confidence', fontsize=LABEL_FONTSIZE);  ax.set_ylabel('Mean opinion component', fontsize=LABEL_FONTSIZE)
    ax.set_title('Mean Trust Opinion per Confidence Bin', fontsize=TITLE_FONTSIZE)
    ax.legend(fontsize=LABEL_FONTSIZE - 1);  ax.set_xlim(-0.02, 1.02);  ax.set_ylim(0, 1);  ax.grid(linestyle='--', alpha=0.3)
    fig.suptitle(f'{dataset_name} — Dynamic trust assessment (epoch {last_epoch}, after calibration)', fontsize=TITLE_FONTSIZE)
    os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, f'{dataset_name}_dynamic.pdf'), bbox_inches='tight')
    plt.show();  plt.close(fig)
    print(f'Saved {dataset_name}_dynamic.pdf')


def print_ece_table(directories):
    print('\n% ECE summary table')
    print(r'\begin{tabular}{lccc}')
    print(r'\toprule')
    print(r'\textbf{Dataset} & \textbf{Accuracy} & \textbf{ECE (pre)} & \textbf{ECE (post)} \\')
    print(r'\midrule')
    for ds_name, directory in directories.items():
        epoch_data = list_epoch_files(directory)
        if not epoch_data: continue
        _, bef_path, aft_path = epoch_data[-1]
        d_bef = pd.read_csv(bef_path);  d_aft = pd.read_csv(aft_path)
        n_classes   = sum(1 for c in d_bef.columns if c.startswith('Class_') and c.endswith('_Probability'))
        prob_cols   = [f'Class_{i}_Probability' for i in range(n_classes)]
        true_labels = d_bef['True Label'].values
        acc     = (d_bef[prob_cols].values.argmax(axis=1) == true_labels).mean()
        ece_bef = compute_ece(d_bef);  ece_aft = compute_ece(d_aft)
        print(f'{ds_name:12s} & {acc:.3f} & {ece_bef:.3f} & {ece_aft:.3f} \\\\')
    print(r'\bottomrule')
    print(r'\end{tabular}')


print('Plotting functions ready.')

## Run analysis & generate figures

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)
ds_name  = DATASET.upper()
datasets = {ds_name: PRED_DIR}

plot_trust_all(PRED_DIR, FIG_DIR, ds_name, N_CLUSTERS)
plot_loss_curves(PRED_DIR, FIG_DIR, ds_name)
plot_trust_and_ece(PRED_DIR, FIG_DIR, ds_name, N_CLUSTERS)
plot_cluster_variation(PRED_DIR, FIG_DIR, ds_name)
plot_dynamic_assessment(PRED_DIR, FIG_DIR, ds_name, N_CLUSTERS)
print_ece_table(datasets)

In [ ]:
# ── Download figures as a zip (if not using Drive) ────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('/content/figures', 'zip', FIG_DIR)
files.download('/content/figures.zip')